# Week 42, Day 4: Reversing a linked list

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w042d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

head = Node(1, Node(2, Node(3)))
prev, curr = None, head
while curr is not None:
    nxt = curr.next
    curr.next = prev
    prev, curr = curr, nxt

out, node = [], prev
while node is not None:
    out.append(node.value)
    node = node.next
print(out, head.value, head.next)

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def reverse(head):
    prev, curr = None, head
    while curr is not None:
        nxt = curr.next          # 1. save the rest
        curr.next = prev         # 2. turn this node round
        prev = curr              # 3. it's now the front of the reversed part
        curr = nxt               # 4. carry on with the rest
    return prev

print(show(reverse(from_list([1, 2, 3, 4]))), show(reverse(from_list([7]))), show(reverse(None)))
big = reverse(from_list(list(range(100_000))))
print(big.value, big.next.value)

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def reverse_rec(head):
    if head is None or head.next is None:       # nothing, or one node: already reversed
        return head
    new_head = reverse_rec(head.next)           # reverse the rest; head.next is now its last node
    head.next.next = head                       # hook head on after it
    head.next = None                            # head is the new end
    return new_head

print(show(reverse_rec(from_list([1, 2, 3, 4]))))
try:
    reverse_rec(from_list(list(range(5000))))
except RecursionError:
    print("RecursionError for 5,000 nodes")

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def from_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def show(head):
    out = []
    while head is not None:
        out.append(head.value)
        head = head.next
    return out

def reverse_section(head, i, j):
    dummy = Node(None, head)
    before = dummy
    for _ in range(i):
        before = before.next
    first = before.next                 # will be the last of the reversed section
    prev, curr = None, first
    for _ in range(j - i):
        nxt = curr.next
        curr.next = prev
        prev, curr = curr, nxt
    before.next = prev                  # the section's new first node
    first.next = curr                   # reconnect to the node after the section
    return dummy.next

print(show(reverse_section(from_list(list("abcdefg")), 2, 5)))
print(show(reverse_section(from_list(list("abcdefg")), 0, 7)))

In [ ]:
class Node:
    def __init__(self, value):
        self.value, self.prev, self.next = value, None, None

def build(values):
    head = tail = None
    for v in values:
        node = Node(v)
        if tail is None:
            head = tail = node
        else:
            node.prev, tail.next, tail = tail, node, node
    return head, tail

def reverse_doubly(head):
    node, new_head = head, head
    while node is not None:
        node.prev, node.next = node.next, node.prev
        new_head = node
        node = node.prev                 # prev now holds what was next
    return new_head

head, tail = build("abcd")
head = reverse_doubly(head)
out, node = [], head
while node is not None:
    out.append(node.value)
    node = node.next
print(out, head.prev, head.next.prev.value)

> **Key idea.** Reverse a singly linked list with three pointers: save `nxt = curr.next`, point `curr.next` at `prev`, then move `prev` and `curr` on; when `curr` is `None`, `prev` is the new head. That's O(n) time and O(1) space. Recursively: reverse the rest, then `head.next.next = head` and `head.next = None`; elegant, but one call per node, so Python's recursion limit makes the loop safer for long lists. To reverse a section, hold the node before it (a dummy head helps), reverse `j - i` nodes, and reconnect both ends. A doubly linked list reverses by swapping each node's links. Reversal is a building block: palindromes, reordering and k-groups all use it.

### Exercise: Reverse with three pointers

Write `reverse(head)`, which reverses a singly linked list in place with a loop and returns the new head. Don't create any nodes, and don't use recursion; the checks reverse a list of 50,000 nodes.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef nodes(head):\n    out = []\n    while head is not None:\n        out.append(head)\n        head = head.next\n    return out\ndef test_reverse():\n    \"reversing\"\n    h = build([1, 2, 3, 4])\n    originals = nodes(h)\n    r = reverse(h)\n    assert [n.value for n in nodes(r)] == [4, 3, 2, 1] and nodes(r) == originals[::-1], \"the same nodes, relinked\"\n    assert reverse(None) is None and reverse(build([5])).value == 5\n    big = reverse(build(list(range(50_000))))\n    assert big.value == 49_999 and big.next.value == 49_998")

### Exercise: Reverse recursively

Write `reverse_rec(head)`, which reverses a list **recursively**, with no loops: reverse the rest, then attach the head at the end. Return the new head. (The checks use lists of up to 500 nodes.)

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse_rec(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head, limit=1000):\n    out = []\n    while head is not None and len(out) < limit:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_rec():\n    \"recursively\"\n    assert show(reverse_rec(build([1, 2, 3]))) == [3, 2, 1] and reverse_rec(None) is None and show(reverse_rec(build([\"x\"]))) == [\"x\"]\n    assert show(reverse_rec(build(list(range(500))))) == list(range(499, -1, -1))\n    fn = next(n for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name == \"reverse_rec\")\n    assert not any(isinstance(n, (ast.For, ast.While)) for n in ast.walk(fn)), \"no loops\"")

### Exercise: Reverse a section

Write `reverse_section(head, i, j)`: reverse the nodes in positions `i` to `j - 1` (half-open, from 0), leaving the rest in place, and return the head. `i == j` changes nothing. Positions must satisfy `0 <= i <= j <= length`, else `IndexError` (and nothing changes). Relink the existing nodes.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse_section(head, i, j):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head, limit=100):\n    out = []\n    while head is not None and len(out) < limit:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_section():\n    \"sections\"\n    rng = random.Random(5)\n    for _ in range(150):\n        vals = list(range(rng.randint(0, 9)))\n        i = rng.randint(0, len(vals))\n        j = rng.randint(i, len(vals))\n        assert show(reverse_section(build(vals), i, j)) == vals[:i] + vals[i:j][::-1] + vals[j:], (vals, i, j)\n    for args in [(0, 4), (2, 1), (-1, 2)]:\n        h = build([1, 2, 3])\n        try:\n            reverse_section(h, *args)\n            assert False, args\n        except IndexError:\n            assert show(h) == [1, 2, 3]")

### Exercise: A palindrome in O(1) space

Write `is_palindrome(head)`: does the list read the same forwards and backwards? Use O(1) extra space: find the middle with slow and fast pointers, reverse the second half, compare, then **reverse it back**, so the list is exactly as it was when the function returns. No lists, strings or other collections.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def is_palindrome(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef nodes(head):\n    out = []\n    while head is not None:\n        out.append(head)\n        head = head.next\n    return out\ndef test_pal():\n    \"palindromes\"\n    rng = random.Random(6)\n    for _ in range(200):\n        n = rng.randint(0, 9)\n        half = [rng.randint(0, 2) for _ in range(n // 2)]\n        vals = half + ([rng.randint(0, 2)] if n % 2 else []) + half[::-1] if rng.random() < 0.5 else [rng.randint(0, 2) for _ in range(n)]\n        h = build(vals)\n        before = nodes(h)\n        assert is_palindrome(h) == (vals == vals[::-1]), vals\n        after = nodes(h)\n        assert after == before and [x.value for x in after] == vals, \"put the list back as it was\"\ndef test_space():\n    \"O(1) space\"\n    import ast\n    tree = ast.parse(source)\n    assert not any(isinstance(n, (ast.List, ast.ListComp, ast.Dict, ast.Set, ast.Tuple)) and not isinstance(getattr(n, \"ctx\", None), ast.Store) for n in ast.walk(tree)\n                   if not isinstance(n, ast.Tuple)), \"no lists, dicts or sets\"")

### Exercise: Reverse a doubly linked list

Write `reverse_doubly(head)`: reverse a doubly linked list (no sentinel) by swapping each node's `prev` and `next`, and return the new head. Afterwards, every link must still be matched in both directions.

In [ ]:
class Node:
    def __init__(self, value):
        self.value, self.prev, self.next = value, None, None

def reverse_doubly(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = tail = None\n    for v in values:\n        node = Node(v)\n        if tail is None:\n            head = tail = node\n        else:\n            node.prev, tail.next, tail = tail, node, node\n    return head\ndef test_doubly():\n    \"both directions\"\n    for vals in [list(\"abcd\"), [\"x\"], []]:\n        h = reverse_doubly(build(vals))\n        out, node, last = [], h, None\n        while node is not None:\n            out.append(node.value)\n            assert node.prev is last, \"back links must match\"\n            last, node = node, node.next\n        assert out == vals[::-1]")

### Exercise: Fold a list

Write `fold(head)`: rearrange `1 2 3 4 5 6` into `1 6 2 5 3 4` (first, last, second, second-last, ...) in place, by relinking. Find the middle, cut the list there, reverse the second half, and interleave the two halves. Return the head (which doesn't change).

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def fold(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head, limit=100):\n    out = []\n    while head is not None and len(out) < limit:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_fold():\n    \"folding\"\n    for n in range(0, 9):\n        vals = list(range(1, n + 1))\n        want = []\n        i, j = 0, n - 1\n        while i <= j:\n            want.append(vals[i])\n            if i != j:\n                want.append(vals[j])\n            i, j = i + 1, j - 1\n        assert show(fold(build(vals))) == want, (vals, show(fold(build(vals))))")

### Bug hunt: The rest of the list, lost

This turns `curr` round **before** saving `curr.next`, so after the first step it has no way to reach the rest of the list: this prints `[1]`. Fix it so it prints `[3, 2, 1]`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse(head):
    prev, curr = None, head
    while curr is not None:
        curr.next = prev
        nxt = curr.next
        prev, curr = curr, nxt
    return prev

r = reverse(Node(1, Node(2, Node(3))))
out = []
while r is not None:
    out.append(r.value)
    r = r.next
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"save the rest first\"\n    expect_output(\"[3, 2, 1]\")")

### Bug hunt: Returning the wrong pointer

The loop is right, but when it ends `curr` is `None`, and that's what this returns, so the reversed list seems empty: this prints `[]`. Fix it so it prints `[3, 2, 1]`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse(head):
    prev, curr = None, head
    while curr is not None:
        nxt = curr.next
        curr.next = prev
        prev, curr = curr, nxt
    return curr

r = reverse(Node(1, Node(2, Node(3))))
out = []
while r is not None:
    out.append(r.value)
    r = r.next
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prev is the new head\"\n    expect_output(\"[3, 2, 1]\")")

### Bug hunt: A loop at the end

The recursive reversal hooks each head on behind the reversed rest, but never sets `head.next = None`, so the last two nodes point at each other for ever. The walk below stops after 6 steps, and prints `[3, 2, 1, 2, 1, 2]`. Fix it so it prints `[3, 2, 1]`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse_rec(head):
    if head is None or head.next is None:
        return head
    new_head = reverse_rec(head.next)
    head.next.next = head
    return new_head

r = reverse_rec(Node(1, Node(2, Node(3))))
out = []
while r is not None and len(out) < 6:
    out.append(r.value)
    r = r.next
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"end the list\"\n    expect_output(\"[3, 2, 1]\")")

### Puzzle: Three pointers

Arrange the lines into the iterative reversal. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code.

p = ___   # the extra space used by the three-pointer reversal: "O(1)" or "O(n)"
q = ___   # how many links change when reversing a list of 5 nodes
r = ___   # after reversing 1 -> 2 -> 3 in place, what does the node holding 1 point to: "None" or 2
s = ___   # roughly how many nested calls reverse_rec makes for a 300-node list
t = ___   # reversing positions 1 to 3 (half-open) of a b c d e gives this list of letters

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"O(1)\" and q == 5 and r == \"None\", \"p, q and r\"\n    assert s == 300 and t == [\"a\", \"c\", \"b\", \"d\", \"e\"], \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: reverse in groups

Write `reverse_groups(head, k)`: reverse the list `k` nodes at a time, in place, and return the new head. A final group with fewer than `k` nodes stays in its original order. So `1 2 3 4 5 6 7 8` with `k = 3` becomes `3 2 1 6 5 4 7 8`. `k` of 1 or less changes nothing. Relink the existing nodes, use O(1) extra space and no recursion; the checks use 60,000 nodes.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse_groups(head, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef show(head, limit=100_000):\n    out = []\n    while head is not None and len(out) < limit:\n        out.append(head.value)\n        head = head.next\n    return out\ndef expected(vals, k):\n    if k <= 1:\n        return vals\n    out = []\n    for i in range(0, len(vals), k):\n        group = vals[i:i + k]\n        out += group[::-1] if len(group) == k else group\n    return out\ndef test_groups():\n    \"groups\"\n    assert show(reverse_groups(build(list(range(1, 9))), 3)) == [3, 2, 1, 6, 5, 4, 7, 8]\n    rng = random.Random(7)\n    for _ in range(150):\n        vals = list(range(rng.randint(0, 12)))\n        k = rng.randint(0, 5)\n        assert show(reverse_groups(build(vals), k)) == expected(vals, k), (vals, k)\ndef test_big():\n    \"60,000 nodes\"\n    vals = list(range(60_000))\n    assert show(reverse_groups(build(vals), 7)) == expected(vals, 7)")

## Warm-up

Every arrow turned round, no node was created, and the old head is now the last node, pointing at `None`. The whole trick is in four lines; today takes them apart.

## Three pointers

Reversing a list means making every node point to the one that used to be **before** it. Walking forwards, you need three names:

- `curr`: the node being turned round;
- `prev`: the node it should now point to (the part already reversed);
- `nxt`: the rest of the list, saved **before** `curr.next` is overwritten, or it would be lost.

```text
start:      prev=None    curr=1 -> 2 -> 3 -> None

step 1:     None <- 1    curr=2 -> 3 -> None         (save nxt=2, point 1 at None, move on)
step 2:     None <- 1 <- 2    curr=3 -> None
step 3:     None <- 1 <- 2 <- 3    curr=None          stop: prev is the new head
```

It's O(n) time and O(1) extra space, and it works for the empty list and a single node without any special case. The loop **invariant** (Week 18) makes it easy to trust: at the top of each pass, `prev` is the reversed version of everything already visited, and `curr` is the untouched rest.

## Recursively

Recursion (Week 15) gives a different view: to reverse a list, reverse **everything after the head**, then put the head on the end. The node after the head is about to become the last node of the reversed rest, so `head.next.next = head` hooks the head on behind it:

It's elegant, but it makes one call per node, and Python stops at about 1,000 calls deep (Week 15). So in Python, write list algorithms with loops unless the list is known to be short; recursion is still worth knowing, because it's how many people think about lists, and how languages without loops do it.

## Reversing a section

To reverse only the nodes in positions `i` to `j - 1`, three nodes matter: the node **before** the section (a dummy head handles `i == 0`), the section's first node (which will end up last), and the node **after** it (which the reversed section must reconnect to).

The inner loop is exactly the three-pointer reversal, run for `j - i` steps instead of to the end, and when it stops, `curr` is the node after the section, ready to reconnect.

## Doubly linked lists

In a doubly linked list, reversing is even simpler: swap every node's `prev` and `next`, and swap the head and tail. Each node already knows both neighbours, so there's nothing to save:

And with a sentinel (Day 2), you don't need to reverse at all if you only want to walk backwards: follow `prev` instead.

## Reversal as a tool

Reversal often appears **inside** other algorithms. To check whether a singly linked list is a palindrome using only O(1) extra space: find the middle (Day 1's slow and fast pointers), reverse the second half, compare the two halves node by node, and reverse the second half back so the caller's list is unchanged. (Always put back what you borrow: a function that quietly rearranges its input is a nasty surprise, as Week 37's defensive programming warned.)

Other uses: adding two numbers stored most-significant digit first (reverse both, add as in Day 1's challenge, reverse the result); reordering `1 2 3 4 5 6` into `1 6 2 5 3 4` (split, reverse the second half, interleave); and reversing in groups of k, today's challenge.

## Practice

## Quiz

## Daily challenge

## Before you go

Trace the three-pointer reversal on a four-node list on paper, writing `prev`, `curr` and `nxt` at each step, then check the invariant at the top of each pass. Tomorrow: cycle detection, and Floyd's tortoise and hare, which finds a loop in a list with only two pointers.